In [15]:
import numpy as np
import pandas as pd

n = 400
seed = np.random.randint(1, 65555)
rng = np.random.default_rng(seed)

In [16]:
popularity_weights = {
    'CS2':               1.45,
    'Dota 2':            1.32,
    'League of Legends': 1.55,
    'Valorant':          1.28,
    'Overwatch 2':       0.92,
    'Rainbow Six Siege': 1.05,
    'Rocket League':     0.88,
    'Apex Legends':      1.12,
    'Fortnite':          1.18,
    'StarCraft II':      0.75
}

discipline_list = list(popularity_weights.keys())
discipline_weights = np.array(list(popularity_weights.values()))
discipline_prob = discipline_weights / discipline_weights.sum()

In [17]:
df = pd.DataFrame()

df['discipline_name'] = rng.choice(discipline_list, size=n, p=discipline_prob)

popularity_weight = df['discipline_name'].map(popularity_weights)
offline_prob = 0.20 + 0.5 * (popularity_weight / popularity_weight.max())

df['tournament_type'] = offline_prob.apply(
    lambda p: np.random.choice(['offline', 'online'], p=[p, 1-p])
)

df['hour'] = rng.integers(8, 24, n)
df['day_of_week'] = rng.integers(0, 7, n)
df['month'] = rng.integers(1, 12, n)

def round_prize(x):
    x = np.asarray(x)
    magnitude = np.floor(np.log10(np.maximum(x, 100)))
    divisor = 10 ** np.where(magnitude <= 3, 2,
                   np.where(magnitude == 4, 2 if x < 50000 else 3,
                   np.where(magnitude == 5, 3,
                   np.where(magnitude == 6, 3 if x < 500000 else 4,
                            4))))
    return np.round(x / divisor) * divisor

base_prize = 25000
df['prize_pool'] = (
    base_prize *
    popularity_weight ** 2.1 *
    np.random.lognormal(0, 1.35, n)
).apply(round_prize).astype(int).clip(1500, 2_500_000)

In [18]:
base_registered = (
    80 *
    (df['prize_pool'] / 10000) ** 0.75 *
    popularity_weight ** 1.1 *
    np.where(df['tournament_type'] == 'offline', 1.45, 0.75)
)

noise = rng.lognormal(0, 0.48, n)
season_factor = 1 + 0.15 * np.sin(2 * np.pi * df['month'] / 12)

day_factor = 1.0 + 0.25 * (df['day_of_week'].isin([5, 6]).astype(float))   # Сб + Вс
day_factor += 0.12 * (df['day_of_week'] == 4).astype(float)                # Пт

hour_factor = np.where(
    (df['hour'] >= 18) & (df['hour'] <= 22), 1.18,
    np.where((df['hour'] >= 14) & (df['hour'] <= 23), 1.08,
    np.where(df['hour'] < 12, 0.82, 0.95))
)

df['registered_count'] = np.round(
    base_registered * noise * season_factor * day_factor * hour_factor
).astype(int).clip(15, 12000)

df['actual_attendance'] = 0

# 1. Offline
offline_mask = df['tournament_type'] == 'offline'
df.loc[offline_mask, 'actual_attendance'] = np.round(
    df.loc[offline_mask, 'registered_count'] * rng.uniform(0.68, 0.94, offline_mask.sum())
).astype(int)

# 2. Online
online_mask = ~offline_mask

online_base = np.sqrt(df.loc[online_mask, 'registered_count']) * 0.35 + \
              (df.loc[online_mask, 'prize_pool'] / 15000) ** 0.6

df.loc[online_mask, 'actual_attendance'] = np.round(
    online_base * rng.lognormal(0, 0.55, online_mask.sum())
).astype(int).clip(0, 280)

# 3. Бонус на крупных онлайн-турнирах
large_online_mask = (df['prize_pool'] > 300_000) & online_mask
if large_online_mask.any():
    df.loc[large_online_mask, 'actual_attendance'] = np.round(
        df.loc[large_online_mask, 'actual_attendance'] * rng.uniform(1.3, 2.2, large_online_mask.sum())
    ).astype(int).clip(upper=280)

# 4. Финальные корректировки
df['actual_attendance'] = df['actual_attendance'].clip(0, df['registered_count'])

# Небольшой шум на мелких турнирах
small_mask = df['registered_count'] < 100
if small_mask.any():
    df.loc[small_mask, 'actual_attendance'] = np.round(
        df.loc[small_mask, 'actual_attendance'] * rng.uniform(0.6, 1.4, small_mask.sum())
    ).astype(int).clip(0, df.loc[small_mask, 'registered_count'])

In [19]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent
OUTPUT_PATH = ROOT / "data" / "training" / "attendance_synthetic.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"seed={seed}, rows={len(df)}")
print(df["tournament_type"].value_counts(normalize=True))
# print("actual_attendance:\n", df["actual_attendance"].describe())

seed=56628, rows=400
tournament_type
offline    0.5875
online     0.4125
Name: proportion, dtype: float64
